# Mistral 7B LoRA Fine-Tuning for Lyric Generation

This notebook fine-tunes Mistral 7B with LoRA on T4 GPU using 480 validated lyric training examples from the HERMES research phase.

**Runtime:** ~45 minutes on T4 (Google Colab free tier)

**Memory:** 8GB peak (4-bit quantization + LoRA)


In [ ]:
# Cell 1: Install dependencies
%pip install -q transformers peft bitsandbytes torch accelerate datasets

import os
os.environ['HF_DATASETS_CACHE'] = '/tmp/hf_cache'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

In [ ]:
# Cell 2: Load training data
# Download from GitHub or load locally
training_file = '/content/training_data.jsonl'  # Upload this to Colab

# If not present, download from GitHub
if not os.path.exists(training_file):
    !git clone -q --depth 1 https://github.com/KudbeeZero/kudbee-music.git /tmp/kudbee
    training_file = '/tmp/kudbee/hermes-lyric-server/training/data/training_data.jsonl'

# Load and verify
examples = []
with open(training_file) as f:
    for line in f:
        ex = json.loads(line)
        examples.append(ex)

print(f"Loaded {len(examples)} training examples")
print(f"Example: {examples[0]}")

In [ ]:
# Cell 3: Configure 4-bit quantization + LoRA
model_id = "mistralai/Mistral-7B-Instruct-v0.1"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for LoRA
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Cell 4: Prepare training data
# Format: instruction + response → SFT format
def format_example(ex):
    instruction = ex.get('instruction', '')
    response = ex.get('response', '')
    return {
        "text": f"""<s>[INST] {instruction} [/INST]
{response} </s>"""
    }

formatted = [format_example(ex) for ex in examples]

# Save to temporary file for trainer
train_file = '/tmp/train.jsonl'
with open(train_file, 'w') as f:
    for item in formatted:
        f.write(json.dumps(item) + '\n')

print(f"Prepared {len(formatted)} training examples")

In [ ]:
# Cell 5: Train
training_args = TrainingArguments(
    output_dir="./lora_model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_steps=200,
    max_steps=480,  # 480 examples, 1 epoch
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_file,
    args=training_args,
    peft_config=lora_config,
    dataset_text_field="text",
    tokenizer=tokenizer,
    max_seq_length=512,
)

trainer.train()
print("✅ Training complete")

In [ ]:
# Cell 6: Save adapter + merge to master model
# Save LoRA adapter
model.save_pretrained('./lora_adapter')
print("✅ Saved LoRA adapter")

# Merge LoRA into base model (for inference)
merged_model = model.merge_and_unload()
merged_model.save_pretrained('./mistral_7b_lora_merged')
tokenizer.save_pretrained('./mistral_7b_lora_merged')
print("✅ Saved merged model")

# Download both
print("\nDownload these files:")
print("  - ./lora_adapter/ (small LoRA weights)")
print("  - ./mistral_7b_lora_merged/ (full merged model for inference)")